In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statistics
import matplotlib.pyplot as plt
sns.set(font_scale=1.5)

df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df.pkl")
df = df[["VIX","NewsVolume","VIX return", "NewsVolume return"]]

In [ ]:
df.columns.values

In [ ]:
df["VIX return"].mean()

In [ ]:
df["VIX NR"] = df["VIX return"].mean()
df["NewsVolume NR"] = df["NewsVolume return"].mean()

df["VIX AR"] = df["VIX return"] - df["VIX NR"]
df["NewsVolume AR"] = df["NewsVolume return"] - df["NewsVolume NR"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statistics

sns.set_style('white')
sns.set_palette('dark')

volume_stdev = 2 * statistics.stdev(df["NewsVolume AR"].dropna())
volume_mean = df["NewsVolume AR"].mean()
volume = df.loc[(df["NewsVolume AR"] >= volume_mean + volume_stdev) | (df["NewsVolume AR"] <= volume_mean - volume_stdev)]
x0 = volume_mean - volume_stdev
x1 = volume_mean + volume_stdev
volume_color = "#C44E52"

plt.figure(figsize=(8, 6))
plt.title("The New York Times\nDistribution of News Volume Abnormal Returns\n(Two Standard Deviations Above or Below the Mean)", fontsize=16)
plt.xlabel("News Volume Abnormal Returns", fontsize=14)
plt.ylabel("Density", fontsize=14)

ax = sns.distplot(df['NewsVolume AR'], hist=True, kde=True, color='#388E3C', kde_kws={'linewidth': 2})
kde_x, kde_y = ax.lines[0].get_data()

p1 = plt.axvline(x=x0, color=volume_color, label="Mean $\pm$ 2 SD", linewidth=2)
p2 = plt.axvline(x=x1, color=volume_color, linewidth=2)
ax.fill_between(kde_x, kde_y, where=(kde_x < x0) | (kde_x > x1), interpolate=True, color=volume_color, alpha=0.5)
plt.legend(fontsize=12)

sns.despine()

plt.show()

In [ ]:
df_major = pd.DataFrame(data=volume.index)
df_major.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodlingprep/code/EventStudy/PKL/volume2stdev.pkl")

In [ ]:
# Define the moving average decay factor
decay_factor = 0.9

# Calculate the moving average of the news volume
df['moving_average'] = df["NewsVolume AR"].ewm(alpha=1-decay_factor).mean()


news_above_ma = df[df["NewsVolume AR"] > (df['moving_average'] * 1.1)]

In [ ]:
df['moving_average']

In [ ]:
# Plotting the histogram
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='moving_average', bins=30, color='blue', alpha=0.8, label='Moving Average')

# Customize the plot
sns.set_style("whitegrid")
plt.xlabel("Moving Average of News Volume AR", fontsize=12, fontweight='bold')
plt.ylabel("Frequency", fontsize=12, fontweight='bold')
plt.title("Press-Release\nHistogram of Exponential Moving Average of News Volume AR", fontsize=14, fontweight='bold')
plt.legend()

# Add vertical lines for mean and standard deviation
mean = df['moving_average'].mean()
std = df['moving_average'].std()
plt.axvline(mean, color='red', linestyle='--', linewidth=1.5, label='Mean')
plt.axvline(mean + std, color='orange', linestyle='--', linewidth=1.5, label='Mean + Std')
plt.axvline(mean - std, color='orange', linestyle='--', linewidth=1.5, label='Mean - Std')

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Calculate the mean and standard deviation of the moving average
mean_ma = df['moving_average'].mean()
std_ma = df['moving_average'].std()

# Get the values above one standard deviation from the mean
values_above_std = pd.DataFrame(df[df['moving_average'] > (mean_ma + std_ma)]['moving_average'])

# Print the values
print(values_above_std)
len(values_above_std)

In [ ]:
# Get the values below one standard deviation from the mean
values_below_std = pd.DataFrame(df[df['moving_average'] < (mean_ma - std_ma)]['moving_average'])

# Print the values
print(values_below_std)

In [ ]:
values_above_std.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodlingprep/code/EventStudy/PKL/volumeAbove2stdevMA.pkl")
values_below_std.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodlingprep/code/EventStudy/PKL/volumeBelow2stdevMA.pkl")

In [ ]:
values_above_std

In [ ]:
import numpy as np

# Define the threshold for high news volume
threshold = df['NewsVolume AR'].quantile(0.8)  # Adjust the quantile as desired

# Create a binary variable indicating high or low news volume
df['HighNewsVolume'] = np.where(df['NewsVolume AR'] > threshold, 1, 0)

# Calculate the average VIX returns during high and low news volume periods
average_returns_high = df[df['HighNewsVolume'] == 1]['VIX AR'].mean()
average_returns_low = df[df['HighNewsVolume'] == 0]['VIX AR'].mean()

# Print the results
print(f"Average VIX returns during high news volume periods: {average_returns_high:.4f}")
print(f"Average VIX returns during low news volume periods: {average_returns_low:.4f}")


In [ ]:
# Define the moving average decay factor
decay_factor = 0.9

# Calculate the moving average of the news volume
df['moving_average'] = df["NewsVolume AR"].ewm(alpha=1-decay_factor).mean()

# Define a function to calculate the average daily return based on the event study threshold
def calculate_average_return_above_ma(threshold):
    news_above_ma = df[df["NewsVolume AR"] > (df['moving_average'] * threshold)]
    dates_above_ma = pd.to_datetime(news_above_ma.index).date
    vix_above_ma = df[df.index.isin(dates_above_ma)]
    return vix_above_ma["VIX AR"].mean()

# Define the event study threshold range
thresholds = [1.1, 1.2, 1.3, 1.4, 1.5]

# Calculate the average returns for each threshold
average_returns = [calculate_average_return_above_ma(threshold) for threshold in thresholds]

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(thresholds, average_returns, marker='o', linestyle='-', color='blue')
plt.xlabel("Event Study Threshold")
plt.ylabel("Average Daily Return of VIX")
plt.title("Average Daily Change of VIX for Moving Average Event Study")
plt.grid(True)
plt.tight_layout()
plt.show()